# RAG evaluation — FinVerify

This notebook runs our **implementation** financial advisor: **Claude (`claude-sonnet-4-6`)** augmented with retrieval over an authoritative knowledge base (IRS publications, CFPB guidance, SEC investor.gov, SSA, HealthCare.gov) persisted in a **ChromaDB** vector store.

Pipeline per question:

1. Embed the question with `BAAI/bge-small-en-v1.5`.
2. Retrieve top-k chunks from Chroma (`k=5`), filterable by topic when useful.
3. Build a prompt that includes retrieved passages with inline citation markers.
4. Call Claude with instructions to cite sources and refuse to fabricate when evidence is thin.
5. Grade with the same three signals as the baseline (MC accuracy, embedding similarity, LLM-as-judge) plus a **citation coverage** signal.

## 1. Prerequisites

```bash
python src/knowledge_base/ingest.py    # builds src/knowledge_base/vector_store/
export ANTHROPIC_API_KEY=sk-ant-...     # for Claude generation and the judge
```

In [1]:
import os, sys, json, time, re, platform, subprocess
from pathlib import Path

REPO_URL = "https://github.com/niksharma99/COMS6156FinalProject.git"
REPO_NAME = "COMS6156FinalProject"
# TODO: switch to "main" once the eval/demo branch is merged.
REPO_BRANCH = "add-evaluation-dataset"

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    clone_target = Path("/content") / REPO_NAME
    if not clone_target.exists():
        print(f"Colab detected - cloning {REPO_URL} ({REPO_BRANCH})")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(clone_target)],
            check=True,
        )
    os.chdir(clone_target)
    REPO_ROOT = clone_target
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "dataset").exists():
        REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "dataset").exists():
        raise RuntimeError(
            f"Could not find repo root (no 'dataset/' dir walking up from {Path.cwd()})."
        )

SRC_DIR = str(REPO_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    print("python-dotenv not installed; relying on shell environment for API keys.")

from eval.sampling import sample_from_dataset
from eval.metrics import grade_mc, cosine_similarity, judge_with_claude

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env or your shell."
print("Repo root:", REPO_ROOT)
print("Python:", platform.python_version())
print("In Colab:", IN_COLAB)


Repo root: /content/COMS6156FinalProject
Python: 3.12.13
In Colab: True


## 1b. Build the knowledge base (run if missing)

The Chroma vector store is a build artifact (gitignored). This cell checks for it and runs `src/knowledge_base/ingest.py` to build it if needed. Subsequent runs are no-ops.


In [2]:
# Build the Chroma vector store if it isn't on disk yet. The store lives at
# src/knowledge_base/vector_store/ and is gitignored (~tens of MB of binaries),
# so a fresh clone (or Colab) needs to build it. ingest.py downloads the source
# documents (IRS / CFPB / SEC / SSA / HealthCare.gov), chunks them, embeds with
# BGE-small, and persists a Chroma collection.
VECTOR_DIR = REPO_ROOT / "src" / "knowledge_base" / "vector_store"
INGEST_SCRIPT = REPO_ROOT / "src" / "knowledge_base" / "ingest.py"

def vector_store_built(p):
    return p.exists() and any(p.iterdir())

if vector_store_built(VECTOR_DIR):
    print(f"Vector store found at {VECTOR_DIR.relative_to(REPO_ROOT)} - skipping build.")
else:
    print(f"Vector store missing at {VECTOR_DIR.relative_to(REPO_ROOT)}. Building...")
    print("(downloads + embedding can take ~3-5 minutes on first run)")
    subprocess.run([sys.executable, str(INGEST_SCRIPT)], check=True)
    assert vector_store_built(VECTOR_DIR), "ingest.py finished but vector_store/ is still empty."
    print("Done.")


Vector store missing at src/knowledge_base/vector_store. Building...
(downloads + embedding can take ~3-5 minutes on first run)
Done.


## 2. Build the evaluation set

Default is `MODE = "all"` (all 156 questions) so the run is apples-to-apples with the baseline. Flip to `"sample"` for a quick smoke test (uses the same `seed=7` the baseline uses in sample mode).


In [3]:
# MODE = "all"     -> every question in every topic file (matches the baseline run)
# MODE = "sample"  -> N_PER_DATASET stratified per dataset (fast smoke test, same seed as baseline)
MODE = "all"
N_PER_DATASET = 5

DATASETS = ["standard_questions", "open_ended_hard", "reddit_questions"]
TOPICS = ["budgeting", "credit_and_debt", "insurance", "investing", "retirement", "tax"]

def load_all_from_dataset(name: str) -> list[dict]:
    base = REPO_ROOT / "dataset" / name
    out = []
    for t in TOPICS:
        for it in json.loads((base / f"{t}.json").read_text()):
            out.append({**it, "_dataset": name, "_topic": t})
    return out

items = []
if MODE == "all":
    for ds in DATASETS:
        items.extend(load_all_from_dataset(ds))
else:
    for ds in DATASETS:
        items.extend(sample_from_dataset(ds, N_PER_DATASET, seed=7))

from collections import Counter
print(f"MODE={MODE}  total={len(items)}")
print("  by dataset:", dict(Counter(i['_dataset'] for i in items)))
print("  by type:   ", dict(Counter('MC' if i.get('type')=='multiple_choice' else 'OE' for i in items)))


MODE=all  total=156
  by dataset: {'standard_questions': 42, 'open_ended_hard': 42, 'reddit_questions': 72}
  by type:    {'OE': 126, 'MC': 30}


## 3. Connect to the vector store

In [4]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma = chromadb.PersistentClient(path=str(VECTOR_DIR))
coll = chroma.get_collection("finverify_kb")
print(f"Collection size: {coll.count()} chunks")

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")


def retrieve(question: str, k: int = 5, topic: str | None = None) -> list[dict]:
    q_emb = embedder.encode([question], normalize_embeddings=True)[0].tolist()
    where = {"topic": topic} if topic else None
    res = coll.query(query_embeddings=[q_emb], n_results=k, where=where)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "meta": meta, "distance": dist})
    return hits


# Sanity check
for h in retrieve("What is the difference between a traditional IRA and a Roth IRA?"):
    print(f"  [{h['meta']['publisher']}] {h['meta']['title']}  (d={h['distance']:.3f})")
    print(f"    {h['text'][:120]}")

Collection size: 3460 chunks


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [IRS] Pub 590-A: Contributions to IRAs  (d=0.149)
    IRAs, unless otherwise stated. Likewise, 
references to Roth IRAs generally include Roth SEP IRAs 
but do not include Ro
  [IRS] Pub 590-A: Contributions to IRAs  (d=0.159)
    -B) 
are tax free, and if you choose, you can leave amounts in 
your Roth IRA as long as you live.
Beginning in 2023, SE
  [IRS] Pub 590-A: Contributions to IRAs  (d=0.163)
    e prints on all proofs including departmental reproduction proofs. MUST be removed before printing.
T able I-2. How Are 
  [IRS] Pub 590-B: Distributions from IRAs  (d=0.168)
    Roth IRA or a SIMPLE IRA. T raditional 
IRAs include traditional IRAs that receive employer contri-
butions from SEP arr
  [IRS] Pub 590-B: Distributions from IRAs  (d=0.175)
    E IRAs can be 
designated as Roth IRAs.
Traditional IRA. A traditional IRA is any IRA that isn't a 
Roth IRA or SIMPLE I


## 4. RAG prompt

Each retrieved chunk is assigned an index `[1]`, `[2]`, .... Claude is instructed to cite chunk indices inline for every substantive claim and to say so explicitly when the passages don't contain enough evidence. This gives us a way to measure *citation coverage* later.

In [5]:
SYSTEM_RAG = (
    "You are FinVerify, a careful personal-finance assistant. "
    "When the provided passages are relevant, prefer them and cite the relevant passage indices "
    "inline as [1], [2], etc. after each claim they support. "
    "For widely-established general personal-finance concepts (e.g., the 50/30/20 rule, HMO vs PPO, "
    "term vs whole life insurance, dollar-cost averaging) you may also draw on general knowledge if "
    "the passages don't directly cover them - in that case, answer the question fully without forcing "
    "citations. Never fabricate specific dollar amounts, percentages, contribution limits, or rule "
    "thresholds: those must come from a passage, otherwise say you can't quote a precise figure. "
    "Acknowledge tradeoffs when they exist. Keep answers to 4-7 sentences."
)

def build_rag_messages(item: dict, hits: list[dict]) -> list[dict]:
    ctx_lines = []
    for i, h in enumerate(hits, 1):
        m = h["meta"]
        ctx_lines.append(f"[{i}] ({m['publisher']} - {m['title']})\n{h['text']}")
    context_block = "\n\n".join(ctx_lines)

    if item.get("type") == "multiple_choice":
        options = "\n".join(item["options"])
        user = (
            f"Context:\n{context_block}\n\n"
            f"Question: {item['question']}\n\n{options}\n\n"
            "Choose the best answer based on the passages and your general knowledge of personal finance. "
            "Respond with ONLY the single letter (A, B, C, or D)."
        )
    else:
        user = (
            f"Context:\n{context_block}\n\n"
            f"Question: {item['question']}\n\n"
            "Answer in 4-7 sentences. Cite passages as [n] when they support a claim; "
            "for general personal-finance concepts not covered by the passages you may answer from "
            "general knowledge without citations. Do not invent specific figures."
        )
    return user


## 5. Generate with Claude + RAG

In [7]:
# Resume-safe RAG run. Checkpoints to disk after each item; restarting the kernel
# re-uses completed answers via `id`.
#
# Set FORCE_REDO = True to clear the checkpoint and regenerate all items
# (use this after changing the prompt or model).
FORCE_REDO = False

import anthropic
from anthropic import APIStatusError, APIConnectionError, RateLimitError

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
K = 5

RESULTS_DIR = REPO_ROOT / "src" / "notebooks" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / f"rag_{MODEL}_all.json"

if FORCE_REDO and RESULTS_PATH.exists():
    print(f"FORCE_REDO=True: removing existing {RESULTS_PATH.name}")
    RESULTS_PATH.unlink()

if RESULTS_PATH.exists():
    results = json.loads(RESULTS_PATH.read_text())
    print(f"Resuming: loaded {len(results)} existing results from {RESULTS_PATH.name}")
else:
    results = []
done_ids = {r["id"] for r in results}

remaining = [it for it in items if it["id"] not in done_ids]
total = len(remaining)
print(f"Running RAG on {total} remaining item(s) of {len(items)} total\n", flush=True)

def call_claude_with_retry(msg, system, max_tokens=700, max_attempts=4):
    delay = 2.0
    for attempt in range(1, max_attempts + 1):
        try:
            return client.messages.create(
                model=MODEL, max_tokens=max_tokens,
                system=system,
                messages=[{"role": "user", "content": msg}],
            )
        except (RateLimitError, APIConnectionError, APIStatusError) as e:
            if attempt == max_attempts:
                raise
            print(f"    retry {attempt}/{max_attempts} after {type(e).__name__}: sleeping {delay:.1f}s", flush=True)
            time.sleep(delay)
            delay *= 2

t_run_start = time.time()
for i, item in enumerate(remaining, 1):
    t0 = time.time()
    hits = retrieve(item["question"], k=K, topic=item["_topic"])
    user_msg = build_rag_messages(item, hits)
    resp = call_claude_with_retry(user_msg, SYSTEM_RAG, max_tokens=700)
    answer = resp.content[0].text.strip()
    dt = time.time() - t0
    results.append({
        "id": item["id"], "dataset": item["_dataset"], "topic": item["_topic"],
        "type": item.get("type"), "difficulty": item.get("difficulty"),
        "question": item["question"],
        "reference": item["correct_answer"],
        "candidate": answer,
        "retrieved": [{"title": h["meta"]["title"], "publisher": h["meta"]["publisher"],
                       "source_slug": h["meta"].get("source_slug"), "distance": h["distance"]}
                      for h in hits],
        "latency_sec": round(dt, 2),
    })
    RESULTS_PATH.write_text(json.dumps(results, indent=2))
    n_cites = len(re.findall(r'\[\d+\]', answer))
    print(f"[{i:3d}/{total}] {item['_dataset']}/{item['_topic']}/{item['id']:<12}  {dt:5.1f}s  cites={n_cites}", flush=True)

dt_total = time.time() - t_run_start
print(f"\nDone. {len(results)}/{len(items)} items. This run: {dt_total:.1f}s "
      f"({dt_total/max(total,1):.1f}s/item)")
print(f"Saved to {RESULTS_PATH}")


Running RAG on 156 remaining item(s) of 156 total

[  1/156] standard_questions/budgeting/bud-001         5.6s  cites=2
[  2/156] standard_questions/budgeting/bud-002         1.0s  cites=0
[  3/156] standard_questions/budgeting/bud-003         7.5s  cites=3
[  4/156] standard_questions/budgeting/bud-004         8.2s  cites=4
[  5/156] standard_questions/budgeting/bud-005         7.7s  cites=2
[  6/156] standard_questions/budgeting/bud-006         0.9s  cites=0
[  7/156] standard_questions/credit_and_debt/crd-001         1.1s  cites=0
[  8/156] standard_questions/credit_and_debt/crd-002         1.3s  cites=0
[  9/156] standard_questions/credit_and_debt/crd-003         0.8s  cites=0
[ 10/156] standard_questions/credit_and_debt/crd-004         1.0s  cites=0
[ 11/156] standard_questions/credit_and_debt/crd-005         0.8s  cites=0
[ 12/156] standard_questions/credit_and_debt/crd-006         5.7s  cites=1
[ 13/156] standard_questions/insurance/ins-001         6.8s  cites=0
[ 14/156] standa

## 6. Score

### 6a. MC accuracy

In [8]:
mc_rows = [r for r in results if r["type"] == "multiple_choice"]
for r in mc_rows:
    g = grade_mc(r["candidate"], r["reference"])
    r["mc_correct"] = g["is_correct"]; r["mc_picked"] = g["picked"]
if mc_rows:
    correct = sum(1 for r in mc_rows if r["mc_correct"])
    print(f"MC accuracy: {correct}/{len(mc_rows)} = {correct/len(mc_rows):.1%}")

MC accuracy: 29/30 = 96.7%


### 6b. Embedding similarity (open-ended)

In [9]:
oe_rows = [r for r in results if r["type"] != "multiple_choice"]
for r in oe_rows:
    a, b = embedder.encode([r["reference"], r["candidate"]], normalize_embeddings=True)
    r["embed_cosine"] = cosine_similarity(a, b)
if oe_rows:
    import statistics
    print(f"Mean cosine vs reference: {statistics.mean(r['embed_cosine'] for r in oe_rows):.3f}")

Mean cosine vs reference: 0.898


### 6c. LLM-as-judge rubric

In [10]:
# Resume-safe judge loop: skip OE rows that already have `judge` populated.
judge_client = anthropic.Anthropic()
to_judge = [r for r in oe_rows if "judge" not in r]
print(f"Judging {len(to_judge)} of {len(oe_rows)} OE rows (skipping already-judged)")

for r in to_judge:
    try:
        score = judge_with_claude(r["question"], r["reference"], r["candidate"], client=judge_client)
        r["judge"] = score.to_dict()
    except Exception as e:
        print(f"  {r['id']}: judge failed ({type(e).__name__}) - skipping")
        continue
    print(f"  {r['id']:<12}  f={score.factuality} c={score.completeness} a={score.advice_quality}  mean={score.mean:.2f}")

# Persist the enriched results so judge scores survive a kernel restart.
RESULTS_PATH.write_text(json.dumps(results, indent=2))


Judging 126 of 126 OE rows (skipping already-judged)
  bud-001       f=5 c=5 a=5  mean=5.00
  bud-003       f=5 c=5 a=5  mean=5.00
  bud-004       f=4 c=4 a=5  mean=4.33
  bud-005       f=5 c=5 a=5  mean=5.00
  crd-006       f=5 c=5 a=5  mean=5.00
  ins-001       f=5 c=5 a=5  mean=5.00
  ins-003       f=5 c=5 a=5  mean=5.00
  inv-001       f=5 c=5 a=5  mean=5.00
  inv-003       f=5 c=5 a=5  mean=5.00
  inv-007       f=5 c=5 a=5  mean=5.00
  ret-005       f=5 c=5 a=5  mean=5.00
  tax-008       f=3 c=4 a=4  mean=3.67
  bud-007       f=5 c=5 a=5  mean=5.00
  bud-008       f=5 c=5 a=5  mean=5.00
  bud-009       f=5 c=5 a=5  mean=5.00
  bud-010       f=5 c=5 a=5  mean=5.00
  bud-011       f=5 c=5 a=5  mean=5.00
  bud-012       f=5 c=5 a=5  mean=5.00
  bud-013       f=5 c=5 a=5  mean=5.00
  cd-001        f=5 c=5 a=5  mean=5.00
  cd-002        f=5 c=5 a=5  mean=5.00
  cd-003        f=5 c=5 a=5  mean=5.00
  cd-004        f=5 c=5 a=5  mean=5.00
  cd-005        f=5 c=5 a=5  mean=5.00
  cd-006   

675348

### 6d. Citation coverage

Counts how many of the retrieved passages the model actually cited. Low values mean the model is ignoring retrieval; high values mean it's using the grounded context.

In [11]:
for r in results:
    cited = set(int(x) for x in re.findall(r"\[(\d+)\]", r["candidate"]))
    cited = {c for c in cited if 1 <= c <= len(r["retrieved"])}
    r["n_citations"] = len(cited)
    r["citation_coverage"] = len(cited) / max(len(r["retrieved"]), 1)
import statistics
print(f"Mean citation coverage: {statistics.mean(r['citation_coverage'] for r in results):.2f}")

Mean citation coverage: 0.33


## 7. Summary & side-by-side with the baseline

In [12]:
import pandas as pd
df = pd.DataFrame(results)

def agg(g):
    row = {"n": len(g)}
    if "mc_correct" in g.columns:
        mc = g.dropna(subset=["mc_correct"])
        row["mc_accuracy"] = mc["mc_correct"].mean() if len(mc) else None
    if "embed_cosine" in g.columns:
        oe = g.dropna(subset=["embed_cosine"])
        if len(oe):
            row["mean_cosine"] = oe["embed_cosine"].mean()
    if "judge" in g.columns:
        judged = g.dropna(subset=["judge"])
        if len(judged):
            row["judge_mean"] = judged["judge"].apply(lambda j: j["mean"]).mean()
    if "citation_coverage" in g.columns:
        row["citation_coverage"] = g["citation_coverage"].mean()
    row["mean_latency_sec"] = g["latency_sec"].mean()
    return pd.Series(row)

rag_summary = df.groupby("dataset").apply(agg, include_groups=False)
print("=== RAG ===")
print(rag_summary)

# Align with the baseline (sharded layout: results/baseline_<model>/all.json).
base_dir = REPO_ROOT / "src" / "notebooks" / "results"
baseline_candidates = sorted(base_dir.glob("baseline_*/all.json")) or sorted(base_dir.glob("baseline_*.json"))
if baseline_candidates:
    bp = baseline_candidates[0]
    print(f"\nBaseline file: {bp.relative_to(REPO_ROOT)}")
    base = pd.DataFrame(json.loads(bp.read_text()))
    print("\n=== Baseline ===")
    print(base.groupby("dataset").apply(agg, include_groups=False))
else:
    print("\n(no baseline results on disk - skipping side-by-side)")


=== RAG ===
                       n  mc_accuracy  mean_cosine  judge_mean  \
dataset                                                          
open_ended_hard     42.0          NaN     0.890436    4.936508   
reddit_questions    72.0          NaN     0.898507    4.337963   
standard_questions  42.0     0.966667     0.919914    4.833333   

                    citation_coverage  mean_latency_sec  
dataset                                                  
open_ended_hard              0.485714          7.528571  
reddit_questions             0.377778          8.997778  
standard_questions           0.104762          2.629762  

Baseline file: src/notebooks/results/baseline_Qwen2.5-3B-Instruct/all.json

=== Baseline ===
                       n  mc_accuracy  mean_cosine  judge_mean  \
dataset                                                          
open_ended_hard     42.0          NaN     0.893431    3.809524   
reddit_questions    72.0          NaN     0.870248    2.356481   
standard_

## 8. Persist results

In [13]:
# Shard results the same way the baseline does so comparison tooling is symmetric.
from collections import defaultdict

MODEL_SLUG = MODEL  # already a plain slug
OUT_DIR = REPO_ROOT / "src" / "notebooks" / "results" / f"rag_{MODEL_SLUG}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

(OUT_DIR / "all.json").write_text(json.dumps(results, indent=2))

by_dataset: dict[str, list[dict]] = defaultdict(list)
for r in results: by_dataset[r["dataset"]].append(r)
for ds, rows in by_dataset.items():
    (OUT_DIR / f"{ds}.json").write_text(json.dumps(rows, indent=2))

by_dt: dict[tuple[str, str], list[dict]] = defaultdict(list)
for r in results: by_dt[(r["dataset"], r["topic"])].append(r)
topic_dir = OUT_DIR / "by_topic"; topic_dir.mkdir(exist_ok=True)
for (ds, tp), rows in by_dt.items():
    (topic_dir / f"{ds}__{tp}.json").write_text(json.dumps(rows, indent=2))

by_type: dict[str, list[dict]] = defaultdict(list)
for r in results:
    key = "multiple_choice" if r.get("type") == "multiple_choice" else "open_ended"
    by_type[key].append(r)
for k, rows in by_type.items():
    (OUT_DIR / f"type_{k}.json").write_text(json.dumps(rows, indent=2))

print(f"Wrote {len(results)} results to {OUT_DIR}")
print(f"  datasets: {sorted(by_dataset)}")
print(f"  topic shards: {len(by_dt)}")
print(f"  types: {sorted(by_type)}")


Wrote 156 results to /content/COMS6156FinalProject/src/notebooks/results/rag_claude-sonnet-4-6
  datasets: ['open_ended_hard', 'reddit_questions', 'standard_questions']
  topic shards: 18
  types: ['multiple_choice', 'open_ended']
